# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mayurkharche01/Internship-starter-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

This notebook builds a transparent baseline action score for content refresh prioritization.

The rule uses only observable signals validated in the signal audit. Product flags, label-derived fields, and future-window outcomes are excluded.

The output is a ranked decision-support queue for human review.

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
from IPython.display import display

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
current = Path.cwd().resolve()

repo_root = None

for candidate in [current] + list(current.parents):
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the FlyRank repository root."
    )

print("Repository root:")
print(repo_root)

Repository root:
C:\Users\Ashok\Desktop\Mayur\internship-local-project\Internship-starter-flyrank


In [4]:
DATA_PATH = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded.
Rows: 30000
Columns: 44


In [5]:
required_columns = [
    "content_id",
    "days_since_last_update",
    "impressions_90d"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

print("All baseline columns are available.")

All baseline columns are available.


## 1. My rule and its reason codes

### Baseline rule

I will rank content using two observable signals:

1. **Staleness** — `days_since_last_update`
2. **Visibility** — `impressions_90d`

The score gives more weight to staleness and a smaller weight to visibility.

The rule is:

- +2 points when `days_since_last_update >= 180`
- +1 point when `impressions_90d >= 500`

Therefore the maximum score is 3.

### Action labels

- **REFRESH_REVIEW** — score = 3
- **REVIEW** — score = 1 or 2
- **MONITOR** — score = 0

### Reason codes

Each row receives exactly ONE reason code:

- `STALE_VISIBLE` — stale and visible
- `STALE` — stale but not highly visible
- `VISIBLE` — highly visible but not stale
- `OTHER` — neither condition is met

The score is a transparent baseline for decision-support, not an automatic content decision.

In [6]:
baseline = df.copy()

print("Rows entering baseline:", len(baseline))

Rows entering baseline: 30000


In [7]:
baseline = baseline.dropna(
    subset=[
        "days_since_last_update",
        "impressions_90d"
    ]
).copy()

print("Rows after required-field check:", len(baseline))

Rows after required-field check: 30000


In [8]:
baseline["staleness_component"] = (
    baseline["days_since_last_update"] >= 180
).astype(int)

baseline["visibility_component"] = (
    baseline["impressions_90d"] >= 500
).astype(int)

display(
    baseline[
        [
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "staleness_component",
            "visibility_component"
        ]
    ].head(10)
)

,content_id,days_since_last_update,impressions_90d,staleness_component,visibility_component
0,content_304f48230142,20,3803,0,1
1,content_a1fb4e703a9e,25,15320,0,1
2,content_9aa793d4d895,20,12581,0,1
3,content_331d6c4de07b,22,11751,0,1
4,content_d99b7a2d90ca,14,19140,0,1
5,content_d4084a4bc775,20,3970,0,1
6,content_9a34b442b552,20,20,0,0
7,content_a63219c6e95a,22,1724,0,1
8,content_5e6c160719bc,20,32574,0,1
9,content_c27558df2b0c,104,1240,0,1


In [9]:
baseline["score"] = (
    2 * baseline["staleness_component"]
    + 1 * baseline["visibility_component"]
)

print("Score distribution:")
display(
    baseline["score"]
    .value_counts()
    .sort_index()
)

Score distribution:


score
0    13117
1    16709
2      157
3       17
Name: count, dtype: int64

In [10]:
def assign_reason_code(row):
    if (
        row["staleness_component"] == 1
        and row["visibility_component"] == 1
    ):
        return "STALE_VISIBLE"
    
    elif row["staleness_component"] == 1:
        return "STALE"
    
    elif row["visibility_component"] == 1:
        return "VISIBLE"
    
    else:
        return "OTHER"


baseline["reason_code"] = baseline.apply(
    assign_reason_code,
    axis=1
)

display(
    baseline[
        [
            "content_id",
            "score",
            "reason_code"
        ]
    ].head(10)
)

,content_id,score,reason_code
0,content_304f48230142,1,VISIBLE
1,content_a1fb4e703a9e,1,VISIBLE
2,content_9aa793d4d895,1,VISIBLE
3,content_331d6c4de07b,1,VISIBLE
4,content_d99b7a2d90ca,1,VISIBLE
5,content_d4084a4bc775,1,VISIBLE
6,content_9a34b442b552,0,OTHER
7,content_a63219c6e95a,1,VISIBLE
8,content_5e6c160719bc,1,VISIBLE
9,content_c27558df2b0c,1,VISIBLE


In [11]:
def assign_action(score):
    if score == 3:
        return "REFRESH_REVIEW"
    
    elif score in [1, 2]:
        return "REVIEW"
    
    else:
        return "MONITOR"


baseline["action"] = baseline["score"].apply(
    assign_action
)

print("Action distribution:")

display(
    baseline["action"]
    .value_counts()
)

Action distribution:


action
REVIEW            16866
MONITOR           13117
REFRESH_REVIEW       17
Name: count, dtype: int64

### Why this rule?

The rule is intentionally simple.

Staleness receives the larger weight because the refresh problem is directly concerned with content freshness. Visibility receives a smaller weight because it represents the potential scale of the opportunity.

The purpose is not to predict whether a refresh will definitely succeed. The purpose is to create a transparent queue that a content team can review.

## 2. Build the ranked queue

I now rank every eligible content item using the baseline score.

Higher scores receive higher priority. Ties are ordered using observed impressions so that more visible pages appear first within the same score.

In [12]:
baseline = baseline.sort_values(
    by=[
        "score",
        "impressions_90d"
    ],
    ascending=[
        False,
        False
    ]
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1

display(
    baseline[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d"
        ]
    ].head(20)
)

,rank,content_id,score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,3,STALE_VISIBLE,REFRESH_REVIEW,194,61678
1,2,content_7368877ea310,3,STALE_VISIBLE,REFRESH_REVIEW,194,59472
2,3,content_1bfaa38ff26c,3,STALE_VISIBLE,REFRESH_REVIEW,194,25715
3,4,content_0a91db491d14,3,STALE_VISIBLE,REFRESH_REVIEW,193,13299
4,5,content_5feee3994adb,3,STALE_VISIBLE,REFRESH_REVIEW,194,7812
5,6,content_c2d929d83eaa,3,STALE_VISIBLE,REFRESH_REVIEW,193,7558
6,7,content_b16bd7307b39,3,STALE_VISIBLE,REFRESH_REVIEW,194,4590
7,8,content_fe16a55cd13d,3,STALE_VISIBLE,REFRESH_REVIEW,194,4556
8,9,content_ecb6215e79fd,3,STALE_VISIBLE,REFRESH_REVIEW,194,4429
9,10,content_928af3e22c80,3,STALE_VISIBLE,REFRESH_REVIEW,193,1697


In [13]:
print("Total ranked rows:", len(baseline))

print("\nScore counts:")
display(
    baseline["score"]
    .value_counts()
    .sort_index()
)

print("\nReason-code counts:")
display(
    baseline["reason_code"]
    .value_counts()
)

print("\nAction counts:")
display(
    baseline["action"]
    .value_counts()
)

Total ranked rows: 30000

Score counts:


score
0    13117
1    16709
2      157
3       17
Name: count, dtype: int64


Reason-code counts:


reason_code
VISIBLE          16709
OTHER            13117
STALE              157
STALE_VISIBLE       17
Name: count, dtype: int64


Action counts:


action
REVIEW            16866
MONITOR           13117
REFRESH_REVIEW       17
Name: count, dtype: int64

In [14]:
assert baseline["rank"].is_unique
assert baseline["rank"].min() == 1
assert baseline["rank"].max() == len(baseline)

assert baseline["score"].notna().all()
assert baseline["reason_code"].notna().all()
assert baseline["action"].notna().all()

print("Ranking checks passed.")

Ranking checks passed.


In [15]:
output_dir = repo_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

output_columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_90d"
]

baseline[output_columns].to_csv(
    output_path,
    index=False
)

print("CSV written successfully:")
print(output_path)

print("\nRows written:", len(baseline))

CSV written successfully:
C:\Users\Ashok\Desktop\Mayur\internship-local-project\Internship-starter-flyrank\work\outputs\baseline_action_score.csv

Rows written: 30000


## 3. Top-20 review

I review the highest-ranked 20 content items individually.

For each item I record:

- the action;
- the reason code;
- why it ranked highly;
- what could make the recommendation wrong.

The purpose is to test whether the rule is producing sensible recommendations rather than trusting the score blindly.

In [16]:
top20 = baseline.head(20).copy()

display(
    top20[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d"
        ]
    ]
)

,rank,content_id,score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,3,STALE_VISIBLE,REFRESH_REVIEW,194,61678
1,2,content_7368877ea310,3,STALE_VISIBLE,REFRESH_REVIEW,194,59472
2,3,content_1bfaa38ff26c,3,STALE_VISIBLE,REFRESH_REVIEW,194,25715
3,4,content_0a91db491d14,3,STALE_VISIBLE,REFRESH_REVIEW,193,13299
4,5,content_5feee3994adb,3,STALE_VISIBLE,REFRESH_REVIEW,194,7812
5,6,content_c2d929d83eaa,3,STALE_VISIBLE,REFRESH_REVIEW,193,7558
6,7,content_b16bd7307b39,3,STALE_VISIBLE,REFRESH_REVIEW,194,4590
7,8,content_fe16a55cd13d,3,STALE_VISIBLE,REFRESH_REVIEW,194,4556
8,9,content_ecb6215e79fd,3,STALE_VISIBLE,REFRESH_REVIEW,194,4429
9,10,content_928af3e22c80,3,STALE_VISIBLE,REFRESH_REVIEW,193,1697


### Top-20 review

| Rank | Action | Reason code | Confidence note | What would make it wrong? |
|---:|---|---|---|---|
| 1 | REFRESH_REVIEW | STALE_VISIBLE | High score because the page is both stale and highly visible. | The page may already be accurate and current despite its age. |
| 2 | REFRESH_REVIEW | STALE_VISIBLE | Both baseline signals support review priority. | The observed visibility may not represent a sustainable opportunity. |
| 3 | REFRESH_REVIEW | STALE_VISIBLE | Staleness and visibility jointly increase the score. | A refresh may not address the actual reason for performance. |
| 4 | ... | ... | ... | ... |
| 5 | ... | ... | ... | ... |
| 6 | ... | ... | ... | ... |
| 7 | ... | ... | ... | ... |
| 8 | ... | ... | ... | ... |
| 9 | ... | ... | ... | ... |
| 10 | ... | ... | ... | ... |
| 11 | ... | ... | ... | ... |
| 12 | ... | ... | ... | ... |
| 13 | ... | ... | ... | ... |
| 14 | ... | ... | ... | ... |
| 15 | ... | ... | ... | ... |
| 16 | ... | ... | ... | ... |
| 17 | ... | ... | ... | ... |
| 18 | ... | ... | ... | ... |
| 19 | ... | ... | ... | ... |
| 20 | ... | ... | ... | ... |

## 4. Weak picks + leakage check

I inspect lower-ranked positive recommendations to identify cases where the simple rule may over-prioritize content.

I also verify that the score does not use product flags, label-derived fields, or future-window outcomes.

In [17]:
weak_picks = baseline[
    baseline["score"].isin([1, 2])
].tail(10)

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d"
        ]
    ]
)

,rank,content_id,score,reason_code,action,days_since_last_update,impressions_90d
16873,16874,content_cd892ad205d3,1,VISIBLE,REVIEW,22,500
16874,16875,content_ca56885db527,1,VISIBLE,REVIEW,20,500
16875,16876,content_829eba7c571f,1,VISIBLE,REVIEW,104,500
16876,16877,content_dc34dac6e5f4,1,VISIBLE,REVIEW,20,500
16877,16878,content_56306f73f70e,1,VISIBLE,REVIEW,20,500
16878,16879,content_d61bf94dfdfb,1,VISIBLE,REVIEW,8,500
16879,16880,content_09cea12992eb,1,VISIBLE,REVIEW,22,500
16880,16881,content_4f35dec82501,1,VISIBLE,REVIEW,104,500
16881,16882,content_8bac7191c859,1,VISIBLE,REVIEW,20,500
16882,16883,content_c2e45b46a329,1,VISIBLE,REVIEW,104,500


### Weak-pick observations

The weaker positive picks show where the baseline can be uncertain.

For example, a page can be old without necessarily needing a refresh, and a page can have high visibility without having a clear content-quality problem.

These cases reinforce that the score should be used as a review-priority queue rather than an automatic decision.

In [18]:
forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "health_score",
    "needs_ctr_fix",
    "is_quick_win",
    "needs_engagement_fix",
    "is_underperformer",
    "is_declining",
    "is_initial_refresh_candidate"
]

baseline_feature_columns = [
    "days_since_last_update",
    "impressions_90d"
]

leaked_features_used = [
    col
    for col in baseline_feature_columns
    if col in forbidden_columns
]

print("Baseline feature columns:")
print(baseline_feature_columns)

print("\nForbidden fields:")
print(forbidden_columns)

print("\nForbidden fields used:")
print(leaked_features_used)

assert len(leaked_features_used) == 0

print("\nLeakage check passed.")

Baseline feature columns:
['days_since_last_update', 'impressions_90d']

Forbidden fields:
['trend_direction', 'trend_pct', 'health_score', 'needs_ctr_fix', 'is_quick_win', 'needs_engagement_fix', 'is_underperformer', 'is_declining', 'is_initial_refresh_candidate']

Forbidden fields used:
[]

Leakage check passed.


In [19]:
print("Output columns:")
print(output_columns)

forbidden_output_columns = [
    col
    for col in output_columns
    if col in forbidden_columns
]

print("\nForbidden output fields:")
print(forbidden_output_columns)

assert len(forbidden_output_columns) == 0

print("Output leakage check passed.")

Output columns:
['rank', 'content_id', 'score', 'reason_code', 'action', 'days_since_last_update', 'impressions_90d']

Forbidden output fields:
[]
Output leakage check passed.


### Leakage conclusion

The baseline score uses only `days_since_last_update` and `impressions_90d`, which are observable signals available in the dataset.

I did not use future-window outcomes, product decision flags, or label-derived fields in the score.

The resulting queue is therefore a transparent decision-support baseline.

## Self-check

- [ ] Every section above is filled — Markdown thinking and supporting code
- [ ] The notebook runs top to bottom with no errors
- [ ] No client names, URLs, or private queries anywhere
- [ ] Claims use careful words: observed, measured, directional, decision-support
- [ ] No future-window inputs used
- [ ] No label-derived inputs used
- [ ] No product flags used as features
- [ ] One transparent score is used
- [ ] Each row has exactly one reason code
- [ ] Each row has exactly one action label
- [ ] The entire queue is ranked
- [ ] Top 20 rows were reviewed individually
- [ ] Weak picks were discussed
- [ ] Leakage checks passed
- [ ] `work/outputs/baseline_action_score.csv` was generated
- [ ] CSV remains uncommitted if leak-guard excludes it
- [ ] Notebook committed under `work/notebooks/`